# Broadband burst CNN training

Notebook for training simple PyTorch CPU baselines on the manually labeled burst dataset.

Default mode is `envelope_1d`: a compact 1D CNN over `envelope_db`. Change `MODEL_KIND` to `spectrogram_2d` to train on `spec_db` instead.

In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

SEED = 42
MODEL_KIND = "envelope_1d"  # "envelope_1d" or "spectrogram_2d"
DATASET_DIR = Path("dataset")
BATCH_SIZE = 32
EPOCHS = 40
LEARNING_RATE = 1e-3
NUM_WORKERS = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(6, torch.get_num_threads())))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


device(type='cpu')

In [2]:
LABEL_TO_INDEX = {"no_burst": 0, "burst": 1}
INDEX_TO_LABEL = {value: key for key, value in LABEL_TO_INDEX.items()}

metadata = pd.read_csv(DATASET_DIR / "metadata.csv")
metadata = metadata[metadata["label"].isin(LABEL_TO_INDEX)].copy()
metadata["target"] = metadata["label"].map(LABEL_TO_INDEX).astype(int)

print(f"labeled samples: {len(metadata)}")
display(metadata.groupby(["split", "label"]).size().unstack(fill_value=0))
metadata.head()


labeled samples: 710


label,burst,no_burst
split,,
test,54,54
train,248,248
val,53,53


,sample_id,source_file,channel,segment_index,segment_start,segment_end,sample_rate,time_resolution,frequency_resolution,freq_min,freq_max,npz_path,png_path,label,peak_time,split,label_source,target
0,sample_000000,C:\code\vlf\Broadband_data\raw_data\Broadband_...,ns,0,0.0,2.0,100000.0,0.002,50.0,20000.0,30000.0,samples\sample_000000.npz,review/burst/sample_000000.png,burst,NaN,test,manual,1
1,sample_000001,C:\code\vlf\Broadband_data\raw_data\Broadband_...,ns,1,2.0,4.0,100000.0,0.002,50.0,20000.0,30000.0,samples\sample_000001.npz,review/burst/sample_000001.png,burst,NaN,val,manual,1
2,sample_000002,C:\code\vlf\Broadband_data\raw_data\Broadband_...,ns,2,4.0,6.0,100000.0,0.002,50.0,20000.0,30000.0,samples\sample_000002.npz,review/no_burst/sample_000002.png,no_burst,NaN,val,manual,0
3,sample_000003,C:\code\vlf\Broadband_data\raw_data\Broadband_...,ns,3,6.0,8.0,100000.0,0.002,50.0,20000.0,30000.0,samples\sample_000003.npz,review/burst/sample_000003.png,burst,NaN,train,manual,1
4,sample_000004,C:\code\vlf\Broadband_data\raw_data\Broadband_...,ns,4,8.0,10.0,100000.0,0.002,50.0,20000.0,30000.0,samples\sample_000004.npz,review/burst/sample_000004.png,burst,NaN,val,manual,1


In [3]:
def resolve_npz_path(npz_path: str) -> Path:
    return DATASET_DIR / Path(str(npz_path).replace("\\", "/"))


class BroadbandBurstDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, model_kind: str):
        self.frame = frame.reset_index(drop=True)
        self.model_kind = model_kind

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        data = np.load(resolve_npz_path(row["npz_path"]))

        if self.model_kind == "envelope_1d":
            x = data["envelope_db"].astype(np.float32)
            x = (x - x.mean()) / (x.std() + 1e-6)
            x = x[None, :]
        elif self.model_kind == "spectrogram_2d":
            x = data["spec_db"].astype(np.float32)
            x = (x - x.mean()) / (x.std() + 1e-6)
            x = x[None, :, :]
        else:
            raise ValueError(f"Unknown MODEL_KIND: {self.model_kind}")

        y = np.int64(row["target"])
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)


train_df = metadata[metadata["split"] == "train"].copy()
val_df = metadata[metadata["split"] == "val"].copy()
test_df = metadata[metadata["split"] == "test"].copy()

train_loader = DataLoader(BroadbandBurstDataset(train_df, MODEL_KIND), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(BroadbandBurstDataset(val_df, MODEL_KIND), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(BroadbandBurstDataset(test_df, MODEL_KIND), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

x0, y0 = next(iter(train_loader))
print(x0.shape, y0.shape)


torch.Size([32, 1, 991]) torch.Size([32])


In [4]:
class EnvelopeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, padding=4),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(64, 2))

    def forward(self, x):
        return self.classifier(self.features(x))


class SpectrogramCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.25), nn.Linear(64, 2))

    def forward(self, x):
        return self.classifier(self.features(x))


model = EnvelopeCNN() if MODEL_KIND == "envelope_1d" else SpectrogramCNN()
model = model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
sum(p.numel() for p in model.parameters())


14434

In [5]:
def run_epoch(loader: DataLoader, train: bool):
    model.train(train)
    total_loss = 0.0
    all_y = []
    all_pred = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item()) * len(y)
        all_y.append(y.detach().cpu().numpy())
        all_pred.append(logits.argmax(dim=1).detach().cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    return total_loss / len(loader.dataset), f1_score(y_true, y_pred, zero_division=0), y_true, y_pred


best_state = None
best_val_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_f1, _, _ = run_epoch(val_loader, train=False)
    history.append({"epoch": epoch, "train_loss": train_loss, "train_f1": train_f1, "val_loss": val_loss, "val_f1": val_f1})

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    print(f"epoch {epoch:02d}: train_loss={train_loss:.4f} train_f1={train_f1:.4f} val_loss={val_loss:.4f} val_f1={val_f1:.4f}")

model.load_state_dict(best_state)
pd.DataFrame(history).tail()


epoch 01: train_loss=0.4819 train_f1=0.8033 val_loss=0.5001 val_f1=0.7727
epoch 02: train_loss=0.3419 train_f1=0.8477 val_loss=0.3837 val_f1=0.8235
epoch 03: train_loss=0.3466 train_f1=0.8507 val_loss=0.3892 val_f1=0.8000
epoch 04: train_loss=0.3163 train_f1=0.8646 val_loss=0.3802 val_f1=0.8119
epoch 05: train_loss=0.3407 train_f1=0.8537 val_loss=0.3777 val_f1=0.8235
epoch 06: train_loss=0.3345 train_f1=0.8514 val_loss=0.3849 val_f1=0.8148
epoch 07: train_loss=0.3426 train_f1=0.8624 val_loss=0.3771 val_f1=0.8235
epoch 08: train_loss=0.3228 train_f1=0.8630 val_loss=0.3770 val_f1=0.8235
epoch 09: train_loss=0.3217 train_f1=0.8694 val_loss=0.3985 val_f1=0.8136
epoch 10: train_loss=0.3106 train_f1=0.8671 val_loss=0.3865 val_f1=0.8142
epoch 11: train_loss=0.3325 train_f1=0.8641 val_loss=0.4846 val_f1=0.7874
epoch 12: train_loss=0.3107 train_f1=0.8758 val_loss=0.3782 val_f1=0.8350
epoch 13: train_loss=0.3230 train_f1=0.8577 val_loss=0.4240 val_f1=0.8099
epoch 14: train_loss=0.3068 train_f1=0

,epoch,train_loss,train_f1,val_loss,val_f1
35,36,0.250186,0.905433,0.578459,0.796875
36,37,0.224404,0.902041,0.403100,0.804124
37,38,0.266489,0.893360,4.214555,0.107143
38,39,0.292541,0.878543,0.504275,0.777778
39,40,0.279114,0.894309,0.537540,0.796875


In [6]:
def predict_loader(loader: DataLoader):
    model.eval()
    ys = []
    preds = []
    probs = []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(DEVICE))
            prob = torch.softmax(logits, dim=1).cpu().numpy()
            ys.append(y.numpy())
            preds.append(prob.argmax(axis=1))
            probs.append(prob)
    return np.concatenate(ys), np.concatenate(preds), np.concatenate(probs)


val_y, val_pred, val_prob = predict_loader(val_loader)
test_y, test_pred, test_prob = predict_loader(test_loader)

print("VAL")
print(confusion_matrix(val_y, val_pred))
print(classification_report(val_y, val_pred, target_names=[INDEX_TO_LABEL[0], INDEX_TO_LABEL[1]], zero_division=0))

print("TEST")
print(confusion_matrix(test_y, test_pred))
print(classification_report(test_y, test_pred, target_names=[INDEX_TO_LABEL[0], INDEX_TO_LABEL[1]], zero_division=0))


VAL
[[45  8]
 [ 9 44]]
              precision    recall  f1-score   support

    no_burst       0.83      0.85      0.84        53
       burst       0.85      0.83      0.84        53

    accuracy                           0.84       106
   macro avg       0.84      0.84      0.84       106
weighted avg       0.84      0.84      0.84       106

TEST
[[44 10]
 [ 4 50]]
              precision    recall  f1-score   support

    no_burst       0.92      0.81      0.86        54
       burst       0.83      0.93      0.88        54

    accuracy                           0.87       108
   macro avg       0.88      0.87      0.87       108
weighted avg       0.88      0.87      0.87       108



In [7]:
models_dir = Path("models")
reports_dir = Path("reports")
models_dir.mkdir(exist_ok=True)
reports_dir.mkdir(exist_ok=True)

model_path = models_dir / f"{MODEL_KIND}.pt"
metrics_path = reports_dir / f"{MODEL_KIND}_metrics.json"
predictions_path = reports_dir / f"{MODEL_KIND}_predictions.csv"

torch.save(
    {
        "model_kind": MODEL_KIND,
        "model_state_dict": model.state_dict(),
        "label_to_index": LABEL_TO_INDEX,
        "config": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "seed": SEED},
    },
    model_path,
)

report = {
    "model_kind": MODEL_KIND,
    "train_samples": int(len(train_df)),
    "val_samples": int(len(val_df)),
    "test_samples": int(len(test_df)),
    "best_val_f1": float(best_val_f1),
    "val": {
        "precision": float(precision_score(val_y, val_pred, zero_division=0)),
        "recall": float(recall_score(val_y, val_pred, zero_division=0)),
        "f1": float(f1_score(val_y, val_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(val_y, val_pred).tolist(),
    },
    "test": {
        "precision": float(precision_score(test_y, test_pred, zero_division=0)),
        "recall": float(recall_score(test_y, test_pred, zero_division=0)),
        "f1": float(f1_score(test_y, test_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(test_y, test_pred).tolist(),
    },
    "history": history,
}
metrics_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

predictions = test_df[["sample_id", "source_file", "segment_start", "segment_end", "label", "png_path", "npz_path"]].copy()
predictions["predicted"] = [INDEX_TO_LABEL[int(value)] for value in test_pred]
predictions["prob_no_burst"] = test_prob[:, 0]
predictions["prob_burst"] = test_prob[:, 1]
predictions.to_csv(predictions_path, index=False)

print(f"model: {model_path}")
print(f"metrics: {metrics_path}")
print(f"predictions: {predictions_path}")


model: models\envelope_1d.pt
metrics: reports\envelope_1d_metrics.json
predictions: reports\envelope_1d_predictions.csv


## Manual sample check

Use this block after training, or after loading a saved model into `model`. Set `SAMPLE` to a `sample_id`, `.npz` path, or `.png` path.

In [ ]:
from IPython.display import Image, display

def row_for_sample(sample: str):
    sample = str(sample).strip().strip('"')
    sample_path = Path(sample.replace("\\", "/"))
    sample_id = sample_path.stem if sample_path.suffix else sample

    rows = metadata[metadata["sample_id"] == sample_id]
    if rows.empty:
        rows = metadata[metadata["npz_path"].astype(str).str.replace("\\", "/", regex=False).str.endswith(sample_path.name)]
    if rows.empty:
        rows = metadata[metadata["png_path"].astype(str).str.replace("\\", "/", regex=False).str.endswith(sample_path.name)]
    if rows.empty:
        raise ValueError(f"Sample not found in metadata: {sample}")
    return rows.iloc[0]


def tensor_from_npz(npz_path: Path, model_kind: str):
    data = np.load(npz_path)
    if model_kind == "envelope_1d":
        x = data["envelope_db"].astype(np.float32)
        x = (x - x.mean()) / (x.std() + 1e-6)
        x = x[None, None, :]
    elif model_kind == "spectrogram_2d":
        x = data["spec_db"].astype(np.float32)
        x = (x - x.mean()) / (x.std() + 1e-6)
        x = x[None, None, :, :]
    else:
        raise ValueError(f"Unknown MODEL_KIND: {model_kind}")
    return torch.from_numpy(x)


def predict_sample(sample: str, show_image: bool = True):
    row = row_for_sample(sample)
    npz_path = resolve_npz_path(row["npz_path"])
    png_path = DATASET_DIR / Path(str(row["png_path"]).replace("\\", "/"))

    model.eval()
    with torch.no_grad():
        x = tensor_from_npz(npz_path, MODEL_KIND).to(DEVICE)
        prob = torch.softmax(model(x), dim=1).cpu().numpy()[0]
    predicted = INDEX_TO_LABEL[int(prob.argmax())]

    print(f"sample_id     : {row['sample_id']}")
    print(f"label         : {row['label']}")
    print(f"predicted     : {predicted}")
    print(f"prob_no_burst : {prob[0]:.4f}")
    print(f"prob_burst    : {prob[1]:.4f}")
    print(f"segment       : {row['segment_start']}-{row['segment_end']} s")
    print(f"npz           : {npz_path}")
    print(f"png           : {png_path}")
    if show_image and png_path.exists():
        display(Image(filename=str(png_path)))
    return {"sample_id": row["sample_id"], "label": row["label"], "predicted": predicted, "prob_no_burst": float(prob[0]), "prob_burst": float(prob[1])}


SAMPLE = "sample_000078"  # sample_id, dataset/samples/sample_000078.npz, or dataset/review/.../sample_000078.png
predict_sample(SAMPLE)


## Verify folder check

Put PNG files into `dataset/verify/burst` and `dataset/verify/no_burst`. The filename must keep the original `sample_id`, for example `sample_000549.png`. The notebook will find the corresponding `.npz` through `metadata.csv` and run the trained model.

In [ ]:
VERIFY_DIR = DATASET_DIR / "verify"
(VERIFY_DIR / "burst").mkdir(parents=True, exist_ok=True)
(VERIFY_DIR / "no_burst").mkdir(parents=True, exist_ok=True)

def predict_verify_folder():
    rows = []
    for expected_label in ["no_burst", "burst"]:
        folder = VERIFY_DIR / expected_label
        for png_path in sorted(folder.glob("*.png")):
            result = predict_sample(png_path.name, show_image=False)
            result["expected_label"] = expected_label
            result["verify_png"] = str(png_path)
            result["correct"] = result["predicted"] == expected_label
            rows.append(result)

    if not rows:
        print(f"No PNG files found in {VERIFY_DIR / 'burst'} or {VERIFY_DIR / 'no_burst'}")
        return pd.DataFrame()

    verify = pd.DataFrame(rows)
    display(verify.groupby(["expected_label", "predicted"]).size().unstack(fill_value=0))
    print(f"accuracy: {verify['correct'].mean():.4f} ({verify['correct'].sum()}/{len(verify)})")
    return verify.sort_values("prob_burst", ascending=False)


verify_predictions = predict_verify_folder()
verify_predictions.head(20)
